# 02 — TurboQuant : Compression du KV Cache

**Objectif** : Implémenter TurboQuant (PolarQuant + QJL) pour compresser le KV cache de Qwen3.5-2B fine-tuné.

Pipeline :
```
fine-tune ✅ → TurboQuant compression (ce notebook) → évaluation before/after
```

**Référence** : [TurboQuant (arXiv:2504.19874)](https://arxiv.org/abs/2504.19874) — Google, mars 2026

---
## Architecture TurboQuant

```
Vecteur KV x ∈ Rᵈ (float16, d=128)
        │
        ▼  Rotation aléatoire Π (orthogonale)
   y = Π·x  ~  N(0, 1/d)  par coordonnée
        │
        ▼  Lloyd-Max (b-1 bits) — codebook précalculé
   idx  +  x̃_mse
        │
        ▼  Résidu r = x - x̃_mse
   sign(S·r)  →  QJL 1-bit  →  correction non-biaisée
        │
        ▼
   x̃ = x̃_mse + correction_QJL   (unbiaisé, ~3 bits/coord)
```

**Résultat attendu** : compression ×5 du KV cache, qualité préservée.

## 1. Setup & Drive

In [ ]:
!pip install unsloth datasets transformers accelerate -q

# Cloner le repo pour accéder à src/
import os
if not os.path.exists('/content/efficient-llm-pipeline'):
    !git clone https://github.com/YanissAmz/efficient-llm-pipeline.git /content/efficient-llm-pipeline

import sys
sys.path.append('/content/efficient-llm-pipeline')

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive/efficient-llm-pipeline'
    print('Drive monté.')
except Exception:
    DRIVE_BASE = '/content/efficient-llm-pipeline-data'
    print(f'Drive non disponible — stockage local : {DRIVE_BASE}')

LORA_PATH = f'{DRIVE_BASE}/models/qwen-gsm8k-lora'
os.makedirs(f'{DRIVE_BASE}/models/turboquant_config', exist_ok=True)

assert os.path.exists(LORA_PATH), f'Adaptateurs LoRA introuvables : {LORA_PATH}'
print(f'Adaptateurs trouvés : {os.listdir(LORA_PATH)}')

## 2. Chargement du modèle fine-tuné

On charge directement depuis le dossier Drive qui contient les adaptateurs LoRA.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LORA_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

vram = torch.cuda.memory_allocated() / 1e9
print(f'Modèle chargé | VRAM : {vram:.2f} GB')

## 3. Import TurboQuant depuis src/

Le code est dans `src/turboquant/` — le notebook importe et utilise, sans redéfinir.

In [ ]:
import sys
sys.path.append('/content')  # permet d'importer src/ depuis Colab

from src.turboquant.polar_quant import TurboQuantMSE, TurboQuantProd, TurboQuantCache, build_codebooks

CODEBOOKS = build_codebooks()
print('Codebooks Lloyd-Max précalculés :')
for b, cb in CODEBOOKS.items():
    print(f'  b={b} ({2**b} niveaux) : {cb.round(3)}')

## 5. Validation mathématique

Avant d'attacher TurboQuant au modèle, on vérifie les deux propriétés théoriques :
1. **MSE** : l'erreur de reconstruction diminue avec b
2. **Unbiaisé** : `E[<y, x̃>] ≈ <y, x>` pour TurboQuantProd

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
DIM    = 128
N      = 1000

x = torch.randn(N, DIM, device=device)
x = x / x.norm(dim=-1, keepdim=True)
y = torch.randn(N, DIM, device=device)

print('=== TurboQuantMSE — erreur de reconstruction ===')
for b in [1, 2, 3, 4]:
    quant = TurboQuantMSE(DIM, b, CODEBOOKS).to(device)
    x_hat = quant(x)
    mse   = (x - x_hat).pow(2).mean().item()
    print(f'  b={b} bits → MSE = {mse:.4f}')

print()
print('=== TurboQuantProd — biais sur produit scalaire ===')
true_dot = (y * x).sum(dim=-1)

for b in [2, 3, 4]:
    quant = TurboQuantProd(DIM, b, CODEBOOKS).to(device)
    _, _, _, x_hat = quant.quantize(x)
    approx_dot = (y * x_hat).sum(dim=-1)

    bias = (approx_dot - true_dot).mean().item()
    rmse = (approx_dot - true_dot).pow(2).mean().sqrt().item()
    print(f'  b={b} bits → biais = {bias:.4f} (attendu ≈ 0) | RMSE = {rmse:.4f}')

## 7. Benchmark — VRAM et latence

On compare baseline vs TurboQuant sur 10 questions GSM8K.
Métriques : VRAM peak, temps de génération, accuracy.

In [ ]:
import time
from datasets import load_dataset

dataset   = load_dataset('openai/gsm8k', 'main', split='test')
QUESTIONS = [dataset[i]['question'] for i in range(10)]
EXPECTED  = [dataset[i]['answer'].split('####')[-1].strip() for i in range(10)]

SYSTEM_PROMPT = (
    'Tu es un assistant mathématique expert. '
    'Décompose chaque problème étape par étape en montrant tous les calculs. '
    "Termine toujours par '#### <réponse>' sur la dernière ligne."
)

HEAD_DIM = model.config.hidden_size // model.config.num_attention_heads
print(f'Head dim : {HEAD_DIM}')

def generate(question, use_turboquant=False, max_new_tokens=256):
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
        {'role': 'user',   'content': [{'type': 'text', 'text': question}]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')

    kwargs = {'input_ids': inputs, 'max_new_tokens': max_new_tokens, 'do_sample': False}
    if use_turboquant:
        kwargs['past_key_values'] = TurboQuantCache(dim=HEAD_DIM, bits=3, codebooks=CODEBOOKS)

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**kwargs)
    elapsed = time.time() - t0
    vram    = torch.cuda.max_memory_reserved() / 1e9

    response = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    return response, elapsed, vram

In [ ]:
results = {'baseline': [], 'turboquant': []}

print('Benchmark sur 10 questions GSM8K...\n')
print(f'{"Q":<4} {"Baseline":>20} {"TurboQuant":>20}')
print(f'{"":4} {"time  vram  acc":>20} {"time  vram  acc":>20}')
print('-' * 50)

for i, (q, expected) in enumerate(zip(QUESTIONS, EXPECTED)):
    resp_b, t_b, vram_b = generate(q, use_turboquant=False)
    resp_t, t_t, vram_t = generate(q, use_turboquant=True)

    ans_b = resp_b.split('####')[-1].strip() if '####' in resp_b else '?'
    ans_t = resp_t.split('####')[-1].strip() if '####' in resp_t else '?'

    results['baseline'].append({'time': t_b, 'vram': vram_b, 'correct': ans_b == expected})
    results['turboquant'].append({'time': t_t, 'vram': vram_t, 'correct': ans_t == expected})

    print(f'Q{i+1:<3} {t_b:.1f}s {vram_b:.2f}GB {"✓" if ans_b==expected else "✗":>3}   '
          f'{t_t:.1f}s {vram_t:.2f}GB {"✓" if ans_t==expected else "✗":>3}')

# Résumé
b_acc = sum(r['correct'] for r in results['baseline'])  / 10
t_acc = sum(r['correct'] for r in results['turboquant']) / 10
b_time = sum(r['time'] for r in results['baseline'])  / 10
t_time = sum(r['time'] for r in results['turboquant']) / 10
b_vram = sum(r['vram'] for r in results['baseline'])  / 10
t_vram = sum(r['vram'] for r in results['turboquant']) / 10

print(f"""
{'='*50}
RÉSULTATS (10 questions GSM8K)
{'='*50}
                  Baseline    TurboQuant 3-bit
Accuracy        : {b_acc*100:.0f}%          {t_acc*100:.0f}%
Temps moyen     : {b_time:.1f}s          {t_time:.1f}s
VRAM peak moy.  : {b_vram:.2f} GB       {t_vram:.2f} GB
Compression     : 1×             {16/3:.1f}× (16→3 bits)
""")

## 8. Sauvegarde de la configuration TurboQuant

On sauvegarde les matrices (rotation + QJL) et les codebooks pour `03_evaluate.ipynb`.

In [ ]:
TURBOQUANT_PATH = f'{DRIVE_BASE}/models/turboquant_config'
os.makedirs(TURBOQUANT_PATH, exist_ok=True)

tq = TurboQuantProd(dim=HEAD_DIM, bits=3, codebooks=CODEBOOKS).cuda()

torch.save({
    'rotation' : tq.mse.rotation.cpu(),
    'S'        : tq.S.cpu(),
    'codebooks': {str(b): torch.tensor(cb) for b, cb in CODEBOOKS.items()},
    'config'   : {'dim': HEAD_DIM, 'bits': 3, 'model': 'Qwen/Qwen3.5-2B'},
    'benchmark': results,
}, f'{TURBOQUANT_PATH}/turboquant.pt')

print(f'Configuration sauvegardée → {TURBOQUANT_PATH}/turboquant.pt')

## Récapitulatif

| Composant | Rôle | Bits |
|---|---|---|
| Rotation Π | Gaussianiser les coordonnées | — |
| Lloyd-Max | Quantification MSE-optimale | b-1 |
| QJL | Correction résidu, non-biaisé | 1 |
| **Total** | **Inner-product unbiaisé** | **b** |

**Suite → `03_evaluate.ipynb`** : évaluation complète before/after sur GSM8K (accuracy, latence, VRAM).

---
*[efficient-llm-pipeline](https://github.com/YanissAmz/efficient-llm-pipeline)*